<a href="https://colab.research.google.com/github/Kafkasyahrial/data-science-2026/blob/main/Pertemuan12_KafkaSyahrial_%5B240401010045%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Pertemuan 12 - Aktivitas Hands-on**

Nama: Kafka Syahrial Fauzan

NIM: 240401010045

Prodi: Informatika - Universitas Siber Asia

Langkah 1: Generate & Eksplorasi Dataset Transaksi

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42) # [cite: 456]
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega'] # [cite: 457, 458]

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk [cite: 459]
transaksi = []
for i in range(50):
    n_item = np.random.randint(2, 6) # [cite: 462]
    transaksi.append(list(np.random.choice(produk, n_item, replace=False))) # [cite: 463]

# Suntikkan pola: Roti sering bersama Selai [cite: 464]
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]: # [cite: 466]
        transaksi[i].append('Selai') # [cite: 467]

print('Contoh transaksi:', transaksi[:3]) # [cite: 468]
print('Jumlah transaksi:', len(transaksi)) # [cite: 469]

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


Langkah 2: One-Hot Encoding Transaksi

In [2]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder() # [cite: 473]
te_ary = te.fit(transaksi).transform(transaksi) # [cite: 475]
df = pd.DataFrame(te_ary, columns=te.columns_) # [cite: 476]

print(df.head()) # [cite: 480]

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


Mengubah daftar transaksi mentah menjadi format matriks biner (One-Hot Encoding) agar dapat diproses oleh algoritma Apriori.

Langkah 3: Cari Frequent Itemset dengan Apriori

In [3]:
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]: # [cite: 485]
    freq = apriori(df, min_support=ms, use_colnames=True) # [cite: 486]
    print(f'min_support={ms}: {len(freq)} itemset ditemukan') # [cite: 487]

# Gunakan min_support yang menghasilkan jumlah itemset wajar [cite: 488]
freq_items = apriori(df, min_support=0.1, use_colnames=True) # [cite: 489]
freq_items = freq_items.sort_values('support', ascending=False) # [cite: 490]
print("\nTop 10 Frequent Itemset:")
print(freq_items.head(10)) # [cite: 491]

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

Top 10 Frequent Itemset:
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Menguji beberapa ambang batas nilai min_support untuk melihat pengaruhnya terhadap jumlah itemset yang berhasil ditemukan.

Langkah 4: Bentuk & Saring Aturan Asosiasi

In [4]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5) # [cite: 496, 497]
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False) # [cite: 498]
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)) # [cite: 499, 500]

         antecedents consequents  support  confidence      lift
9        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
14  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
11      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
8       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
15     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
10     (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
13   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Membentuk aturan asosiasi menggunakan metrik confidence minimal 0.5 dan menyaring hasil berdasarkan nilai lift > 1 untuk mendapatkan pola yang valid secara statistik.

Langkah 5: Rekomender Sederhana dengan Content-Based Filtering

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({ # [cite: 508]
    'produk': produk, # [cite: 509]
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy', 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy'] # [cite: 510, 511]
}) # [cite: 512]

fitur = pd.get_dummies(katalog['kategori']) # [cite: 513]
sim_matrix = cosine_similarity(fitur) # [cite: 513]

def rekomendasi_serupa(nama_produk, top_n=3): # [cite: 518]
    idx = katalog.index[katalog['produk'] == nama_produk][0] # [cite: 519]
    skor = list(enumerate(sim_matrix[idx])) # [cite: 519]
    skor = sorted(skor, key=lambda x: x[1], reverse=True) # [cite: 519, 520]
    skor = [s for s in skor if s[0] != idx][:top_n] # [cite: 521]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist() # [cite: 522]

print('Mirip dengan Roti:', rekomendasi_serupa('Roti')) # [cite: 523]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Membangun sistem rekomendasi produk serupa menggunakan kemiripan kategori berbasis Cosine Similarity.

Langkah 6: Bandingkan Kedua Pendekatan

In [6]:
produk_target = 'Roti' # [cite: 527]

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target [cite: 528]
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)] # [cite: 529, 530]

print('Rekomendasi dari Association Rules:') # [cite: 531]
print(rules_terkait[['consequents', 'lift']].head()) # [cite: 532]

print('\nRekomendasi dari Content-Based:') # [cite: 533]
print(rekomendasi_serupa(produk_target)) # [cite: 533]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115

Rekomendasi dari Content-Based:
['Selai', 'Sereal', 'Susu']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Membandingkan output rekomendasi untuk produk target yang sama (Roti) menggunakan metode Association Rules vs Content-Based Filtering.